# Module 09: CI/CD with GitHub Actions

Automate testing and deployment.

## Learning Objectives

1. Understand CI/CD concepts
2. Create GitHub Actions workflows
3. Use AI to generate workflow files
4. Run tests across platforms

## What is CI/CD?

- **CI (Continuous Integration)**: Automatically test code on every push
- **CD (Continuous Deployment)**: Automatically deploy when tests pass

GitHub Actions runs workflows in response to events (push, PR, schedule).

## Basic Workflow

Create `.github/workflows/test.yml`:

```yaml
name: Tests

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    
    steps:
      - uses: actions/checkout@v4
      
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      
      - name: Install dependencies
        run: |
          pip install -e ".[dev]"
      
      - name: Run tests
        run: pytest --cov
```

## Exercise 1: Your First Workflow (10 min)

Get a GitHub Actions workflow running on your repository.

1. **In your GitHub repo, create the workflow file.** From your local clone:
   ```bash
   mkdir -p .github/workflows
   ```

2. **Create `.github/workflows/hello.yml`** with this content:
   ```yaml
   name: Hello World
   on: push

   jobs:
     greet:
       runs-on: ubuntu-latest
       steps:
         - run: echo "Hello from GitHub Actions!"
         - run: python --version
         - run: pip --version
   ```

3. **Commit and push:**
   ```bash
   git add .github/workflows/hello.yml
   git commit -m "ci: add hello world workflow"
   git push
   ```

4. **Go to the Actions tab** on your GitHub repo page. You should see your workflow running (or already completed).

5. **Click on the run** to see the logs. Answer these questions:
   - Did it pass (green check)?
   - What Python version was available by default?
   - How long did it take?

**Expected outcome:** You have a working GitHub Actions workflow and understand how to view its output.

## AI-Generated Workflows

```
> Create a GitHub Actions workflow that:
  - Runs on push and PR to main
  - Tests on Python 3.10, 3.11, 3.12
  - Tests on Ubuntu, macOS, Windows
  - Runs black and ruff checks
  - Runs pytest with coverage
  - Uploads coverage to Codecov
```

AI generates complete workflow - review for correctness!

## Exercise 2: Add a Test Workflow (10 min)

Create a workflow that actually runs your project's tests.

1. **Create `.github/workflows/tests.yml`:**
   ```yaml
   name: Tests
   on: [push, pull_request]

   jobs:
     test:
       runs-on: ubuntu-latest
       steps:
         - uses: actions/checkout@v4

         - name: Set up Python
           uses: actions/setup-python@v5
           with:
             python-version: '3.11'

         - name: Install package
           run: pip install -e ".[dev]"

         - name: Run tests
           run: pytest
   ```

2. **Commit and push:**
   ```bash
   git add .github/workflows/tests.yml
   git commit -m "ci: add test workflow"
   git push
   ```

3. **Go to the Actions tab** and watch the workflow run.

4. **If it fails** (this is common!), click on the failed job and read the logs. The most common issues:
   - Missing dependencies (add them to your `pyproject.toml` or `requirements.txt`)
   - No `[dev]` extra defined (change to `pip install -e .` and `pip install pytest` separately)
   - Tests that depend on local files or environment variables

5. **Fix and push again.** Each push triggers a new run.

**Expected outcome:** Your tests run automatically on every push. You've debugged at least one CI failure (a rite of passage!).

## Matrix Testing

Test across multiple configurations:

```yaml
jobs:
  test:
    runs-on: ${{ matrix.os }}
    strategy:
      matrix:
        os: [ubuntu-latest, macos-latest, windows-latest]
        python-version: ['3.10', '3.11', '3.12']
    
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      - run: pip install -e ".[dev]"
      - run: pytest
```

## Exercise 3: Matrix Build (10 min)

Test your package across multiple Python versions simultaneously.

1. **Modify your `.github/workflows/tests.yml`** to use a matrix strategy:
   ```yaml
   name: Tests
   on: [push, pull_request]

   jobs:
     test:
       runs-on: ubuntu-latest
       strategy:
         matrix:
           python-version: ["3.10", "3.11", "3.12"]

       steps:
         - uses: actions/checkout@v4

         - name: Set up Python ${{ matrix.python-version }}
           uses: actions/setup-python@v5
           with:
             python-version: ${{ matrix.python-version }}

         - name: Install package
           run: pip install -e ".[dev]"

         - name: Run tests
           run: pytest
   ```

   **Important:** The quotes around `"3.10"` matter! Without them, YAML interprets `3.10` as the number `3.1`.

2. **Commit and push:**
   ```bash
   git add .github/workflows/tests.yml
   git commit -m "ci: test across Python 3.10-3.12"
   git push
   ```

3. **Go to the Actions tab.** You should see **3 parallel jobs** — one for each Python version.

4. **Check the results:** Do all Python versions pass? If one fails but others pass, that's valuable information about compatibility.

**Expected outcome:** You see 3 jobs running in parallel on the Actions tab, and you understand how matrix builds help catch version-specific issues.

## Status Badges

Add to README:

```markdown
![Tests](https://github.com/user/repo/actions/workflows/test.yml/badge.svg)
```

Shows build status at a glance.

## Exercise 4: Add a Badge (5 min)

Show your build status right on your README.

1. **Go to your Actions tab** on GitHub.

2. **Click on your "Tests" workflow** in the left sidebar.

3. **Click the `...` menu** (top right) and select **"Create status badge"**.

4. **Copy the markdown** it generates. It will look something like:
   ```markdown
   ![Tests](https://github.com/your-user/your-repo/actions/workflows/tests.yml/badge.svg)
   ```

5. **Add it to the top of your `README.md`**, commit, and push:
   ```bash
   git add README.md
   git commit -m "docs: add CI status badge"
   git push
   ```

6. **View your README on GitHub** — you should see a green "passing" badge (or red "failing" if tests are broken).

7. **Ask your AI agent:**

   > "What other badges are commonly used in Python project READMEs?"

   You'll likely learn about badges for: coverage, PyPI version, license, Python version support, and downloads.

**Expected outcome:** Your README displays a live build status badge, and you know about other common badges for Python projects.

## Publishing to PyPI

```yaml
name: Publish

on:
  release:
    types: [published]

jobs:
  publish:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - run: pip install build twine
      - run: python -m build
      - run: twine upload dist/*
        env:
          TWINE_USERNAME: __token__
          TWINE_PASSWORD: ${{ secrets.PYPI_TOKEN }}
```

## Exercise: Set Up CI

1. Ask AI to generate a workflow for your package
2. Add it to `.github/workflows/`
3. Push and watch it run
4. Fix any failures
5. Add status badge to README

## Summary

| Concept | Purpose |
|---------|--------|
| Workflow | YAML file defining automation |
| Job | Set of steps that run together |
| Matrix | Test multiple configurations |
| Secrets | Store sensitive data |

## Next Steps

In the next module, we'll learn AI-assisted documentation.